
# Script RvW-tool

#### Ontwikkeld door: Wietse Wierks (HDSR), Rob Tijsen (AGV) & Rafi Senden (AGV)  
In opdracht van: Deltaprogramma Centraal Holland binnen het Maatregelenpakket Wateroverlast. 

**DISCLAIMER:**  
Dit is een werkbestand en nog in ontwikkeling. Signaleer je fouten of onduidelijkheden, neem dan contact op via: wietse.wierks@hdsr.nl  

---

## Introductie

Deze tool is ontwikkeld binnen het Deltaprogramma Centraal Holland voor het maatregelenpakket *Wateroverlast – Traject Ruimte voor Water*.  

De tool bestaat uit twee scripts:
- Voorbewerking van data die gebruikt wordt voor de tool;
- De daadwerkelijke tool waarin de RvW wordt berekend in m³ en m² (dit script).

De invoer en  werking van de functies wordt toegelicht via comments en markdown cellen. In de comments in het script zelf wordt een toelichting gegeven over de werking van de functie. In de markdown cellen wordt uitgelegd per stap waarom deze functie wordt uitgevoerd. 


## Workflow

Het script is opgedeeld in:

### Deel 1 - Bouw volume-oppervlakte dataset per peilvak per polder

### Deel 2 - Aggregeer dataset naar polderniveau

### Deel 3 - Bereken RvW o.b.v. Ontwerpbui T


# Deel 1 - Bouw volume-oppervlakte dataset per peilvak per polder

In [3]:
import arcpy
import os
import numpy as np
import pandas as pd
import re
from tqdm.notebook import tqdm
from pathlib import Path
from arcpy.sa import SetNull 

def maak_veilige_naam(
    naam,
    *,
    target="gdb_object",   # "gdb_object" of "filesystem"
    workspace=None,
    max_len=70
):
    """
    Maakt een veilige naam voor:
    - target="filesystem"  → mappen + .gdb namen
    - target="gdb_object"  → feature classes / tabellen in een GDB

    Zet een tekst om naar een veilige naam voor gebruik in het
    bestandssysteem of binnen een geodatabase.
    Bewerkingen:
        - omzetting naar kleine letters
        - spaties vervangen door underscores
        - ongeldige tekens verwijderen/vervangen
        - voorkomen dat namen met een cijfer beginnen
        - validatie van GDB-objectnamen via arcpy
    """
    s = str(naam).strip().lower()

    # Uniforme normalisatie
    s = s.replace(" ", "_")
    s = re.sub(r"[^\w]", "_", s)   # ook - / \ etc.
    s = re.sub(r"_+", "_", s)

    # Niet beginnen met cijfer
    if s and s[0].isdigit():
        s = f"p_{s}"

    if target == "filesystem":
        return s.strip("_")

    if target == "gdb_object":
        if workspace is None:
            raise ValueError("workspace is verplicht bij target='gdb_object'")

        s = s[:max_len]
        s = s.strip("_")
        return arcpy.ValidateTableName(s, workspace)

    raise ValueError(f"Onbekend target: {target}")

### 1.1 Bereken volume en inundatie per peilvak

**Doel**  
Bepalen van inundatievolumes, inundatieoppervlaktes en de verdeling hiervan over verschillende landgebruikstypen voor alle peilgebieden binnen een polder bij oplopende waterstanden. Hiervoor wordt gebruikgemaakt van de voorbewerkte AHN-rasterdata die in de Setup-fase per polder is samengesteld. Per peilgebied wordt vanuit het geldende peil de waterstand stapsgewijs verhoogd met een vooraf gedefinieerd interval tot een maximale verhoging. Voor iedere waterstandsstap wordt op basis van het AHN bepaald welke delen van het maaiveld overstromen, waarna het inundatieoppervlak en het waterbergingsvolume worden berekend en uitgesplitst naar slootoppervlak en de verschillende landgebruikstypen.

**Output**
- Resultatentabel met inundatievolume en inundatieoppervlak per peilgebied en waterstand
- Oppervlaktes en volumes uitgesplitst naar landgebruiksklasse
- CSV-bestand met tussentijdse resultaten


### 1.1 Achtergrondfunctie: Bereken volume en inundatie per x cm peilstijging

Deze achtergrondfunctie berekent per peilvak in iedere polder o.b.v. de voorbewerkte AHN-raster data stapsgewijs het volume en inundatie op het maaiveld per mogelijke waterstand.


In [4]:
def hypsometrie(elevaties, wls):
    z = np.sort(elevaties.astype(np.float64))
    csum = np.concatenate(([0.0], np.cumsum(z, dtype=np.float64)))
    
    wls = np.asarray(wls, dtype=np.float64)
    
    aantal_cellen = np.searchsorted(z, wls, side="left")
    diepte_som = wls * aantal_cellen - csum[aantal_cellen]
    
    return aantal_cellen, diepte_som
    
def bereken_volumes_fast(
    root_folder,
    fld_waterschap="Waterschap",
    fld_polder="Naam_1",
    fld_id="WS_ID",
    fld_peilgebied="CODE",
    fld_maxpeil="Peil_zo",
    fld_maalpand="Type",
    fld_op_pg="OPP_PG",
    fld_op_pldr="OPP_PLDR",
    peil_type="zo",
    interval=0.02,
    hoogte_x=0.5,
    select_waterschap=None,
    select_polder=None
):
    """
    Berekent inundatievolumes en overstroomde oppervlaktes voor alle
    peilgebieden binnen geselecteerde polders en waterschappen op basis
    van hoogte- en landgebruiksrasters.

    Voor ieder peilgebied wordt de waterstand stapsgewijs verhoogd vanaf
    het opgegeven peil tot een maximale verhoging. Per waterstand worden
    inundatiediepte, inundatieoppervlak en waterbergingsvolume berekend.
    Daarnaast wordt onderscheid gemaakt naar verschillende typen
    landgebruik.

    Parameters
    ----------
    root_folder : str
        Hoofdmap met waterschappen en polders. Per polder worden een
        geodatabase, hoogtebestand en landgebruiksraster verwacht.

    fld_waterschap : str, default "Waterschap"
        Naam van het veld met de waterschapsnaam.

    fld_polder : str, default "Naam_1"
        Naam van het veld met de poldernaam.

    fld_peilgebied : str, default "CODE"
        Unieke identificatie van het peilgebied.

    fld_maxpeil : str, default "Peil_zo"
        Veld met het uitgangspeil waarop de berekeningen starten.

    fld_maalpand : str, default "Type"
        Veld met het type maalpand of afwateringseenheid.

    fld_op_pg : str, default "OPP_PG"
        Veld met de oppervlakte van het peilgebied.

    fld_op_pldr : str, default "OPP_PLDR"
        Veld met de oppervlakte van de polder.

    peil_type : str, default "zo"
        Type peil dat wordt gebruikt bij het selecteren van het
        hoogteraster.

    interval : float, default 0.02
        Waterstandsverhoging per stap in meters.

    hoogte_x : float, default 0.5
        Maximale verhoging van de waterstand boven het uitgangspeil.

    select_waterschap : str, optional
        Indien opgegeven worden uitsluitend polders van dit
        waterschap verwerkt.

    select_polder : str, optional
        Indien opgegeven wordt uitsluitend deze polder verwerkt.

    Returns
    -------
    pandas.DataFrame

    DataFrame met per peilgebied en waterstand:

    - waterschap
    - polder
    - oppervlakte polder
    - peilgebied
    - oppervlakte peilgebied
    - peiltype
    - maalpandtype
    - waterstand (WL)
    - inundatievolume (m³)
    - inundatieoppervlak (m²)

    Daarnaast worden voor de volgende landgebruiksklassen zowel
    oppervlaktes als volumes bepaald:

    - water
    - grasland
    - akker
    - hoogwaardig land -en tuinbouw (afgekort: tuinbouw)
    - bebouwing

    Werkwijze
    ---------
    1. Doorloop alle waterschappen en polders binnen de hoofdmap.
    2. Lees de peilgebiedgeometrie en bijbehorende attributen in.
    3. Lees hoogte- en landgebruiksrasters in.
    4. Clip rasters naar het rekengebied van de polder.
    5. Converteer de rasters naar NumPy-arrays voor snelle verwerking.
    6. Rasteriseer de peilgebieden zodat per cel bekend is bij welk
       peilgebied deze hoort.
    7. Verhoog de waterstand stapsgewijs vanaf het uitgangspeil.
    8. Bereken voor iedere stap:
       - waterdiepte per rastercel
       - totaal inundatievolume
       - totaal inundatieoppervlak
       - oppervlaktes per landgebruikstype
       - volumes per landgebruikstype
    9. Sla tussentijdse resultaten op naar CSV.
    10. Retourneer alle resultaten als pandas DataFrame.

    Opmerkingen
    -----------
    De berekeningen maken gebruik van NumPy-arrays in plaats van
    herhaalde ArcGIS-rasteranalyses per peilgebied. Hierdoor kunnen
    grote aantallen waterstandsberekeningen aanzienlijk sneller worden
    uitgevoerd dan met een volledig ArcGIS-gebaseerde workflow.
    """ 
    
    arcpy.env.addOutputsToMap = False
    arcpy.env.overwriteOutput = True
    arcpy.env.parallelProcessingFactor = "100%"
    
    arcpy.CheckOutExtension("Spatial")

    rd_new = arcpy.SpatialReference(28992)
    arcpy.env.outputCoordinateSystem = rd_new

    resultaten = []
    
    output_csv = os.path.join(root_folder, "tussenresultaten.csv")

    for ws in tqdm(os.listdir(root_folder), desc="Waterschappen"):

        if select_waterschap:
            if maak_veilige_naam(ws, target="filesystem") != \
               maak_veilige_naam(select_waterschap, target="filesystem"):
                continue

        ws_path = os.path.join(root_folder, ws)
        if not os.path.isdir(ws_path):
            continue

        for polder in tqdm(os.listdir(ws_path), desc=f"Polders {ws}", leave=False):

#             try:

                if select_polder:
                    if maak_veilige_naam(polder, target="filesystem") != \
                       maak_veilige_naam(select_polder, target="filesystem"):
                        continue

                polder_path = os.path.join(ws_path, polder)
                if not os.path.isdir(polder_path):
                    continue

                print(f"--- {ws} / {polder} ---")

                ws_norm = maak_veilige_naam(ws, target="filesystem")
                polder_norm = maak_veilige_naam(polder, target="filesystem")

                # --------------------------------------------------
                # Rekengebied
                # --------------------------------------------------
                gdb = os.path.join(polder_path, f"{polder}.gdb")
                fc = os.path.join(gdb, f"rekengebied_{polder}")

                if not arcpy.Exists(fc):
                    print("--- geen rekengebied! ---")
                    continue

                fields = [
                    fld_waterschap,
                    fld_polder,
                    fld_id,
                    fld_peilgebied,
                    fld_maxpeil,
                    fld_maalpand,
                    fld_op_pg,
                    fld_op_pldr
                ]
                
                beschikbare_velden = [f.name for f in arcpy.ListFields(fc)]

                if fld_peilgebied not in beschikbare_velden:
                    print(
                        f"--- Veld '{fld_peilgebied}' niet gevonden "
                        f"voor {polder} -> overslaan ---"
                    )
                    print("Beschikbare velden:")
                    print(beschikbare_velden)
                    continue
                    
                rows = list(arcpy.da.SearchCursor(fc, fields))
                if not rows:
                    continue

                df = pd.DataFrame(rows, columns=[
                    "waterschap", "polder", "ws_id", "peilgebied",
                    "maxpeil", "maalpand",
                    "opp_pg", "opp_pldr"
                ])


#                 print(f"  polder map naam: {polder}")
#                 print(f"  polder veld (uniek): {df['polder'].unique()}")

                df["polder_norm"] = df["polder"].apply(
                    lambda x: maak_veilige_naam(x, target="filesystem")
                )

                polder_norm_check = maak_veilige_naam(polder, target="filesystem")

#                 print(f"  polder_norm map: {polder_norm_check}")
#                 print(f"  polder_norm df (uniek): {df['polder_norm'].unique()}")

                df = df[df["polder_norm"] == polder_norm_check]

#                 print(f"  df lengte na filter: {len(df)}")

                if df.empty:
                    print("--- dataframe leeg -> skip!!! ---")
                    continue

                if df.empty:
                    continue

                # --------------------------------------------------
                # Rasters (met DEBUG)
                # --------------------------------------------------
                hoogte_raster = os.path.join(
                    polder_path,
                    f"{ws_norm}_{polder_norm}_Peil_{peil_type}.tif"
                )

                lu_raster = os.path.join(
                    polder_path,
                    f"{ws_norm}_{polder_norm}_lu.tif"
                )

#                 print(f"  hoogte raster: {hoogte_raster}")
#                 print(f"  landuse raster: {lu_raster}")

                if not os.path.exists(hoogte_raster):
                    print("--- hoogte raster ontbreekt! ---")
                if not os.path.exists(lu_raster):
                    print("--- landuse raster ontbreekt! ---")

                if not (os.path.exists(hoogte_raster) and os.path.exists(lu_raster)):
                    print("--- skip polder! ---")
                    continue
                else:
                    print("--- beide rasters gevonden ---")

                print("--- start berekening ---")

                # --------------------------------------------------
                # Hoogte raster laden (GEEN ProjectRaster tenzij nodig)
                # --------------------------------------------------
#                 print("1 Raster openen")
                ahn = arcpy.Raster(hoogte_raster)
                
#                 print("2 CRS check")
                if ahn.spatialReference.factoryCode != 28992:
                    out_raster = os.path.join(
                        arcpy.env.scratchGDB,
                        f"ahn_{polder_norm}"
                    )

                    ahn = arcpy.management.ProjectRaster(
                        ahn,
                        out_raster,
                        rd_new,
                        "BILINEAR"
                    )

                ahn = arcpy.Raster(ahn)

                cellsize = ahn.meanCellWidth
                cellarea = cellsize ** 2

                # --------------------------------------------------
                # Clip 1x op rekengebied
                # --------------------------------------------------
                layer = f"lyr_{polder_norm}"
                arcpy.MakeFeatureLayer_management(fc, layer)
                
#                 print("4 Set environments")
                arcpy.env.snapRaster = ahn
                arcpy.env.extent = layer
                arcpy.env.cellSize = ahn
                
#                 print("5 ExtractByMask START")
                ahn_clip = arcpy.sa.ExtractByMask(ahn, layer)
                
#                 print("6 ExtractByMask GEREED")
                ahn_clip = arcpy.Raster(ahn_clip)
                
#                 print("7 Raster info")
#                 print("width =", ahn_clip.width)
#                 print("height =", ahn_clip.height)
                
                # NumPy conversie
                pixel_type = ahn_clip.pixelType
                is_integer = not pixel_type.startswith("F")
                
#                 print("8 RasterToNumPyArray START")
                
                NODATA = -99999
               
                try:
                
                    if is_integer:

                        ahn_arr = arcpy.RasterToNumPyArray(
                            ahn_clip,
                            nodata_to_value=NODATA
                        ).astype(np.float32)

                        ahn_arr[ahn_arr == NODATA] = np.nan

                        # mm -> m
                        ahn_arr /= 1000.0

                    else:

                        ahn_arr = arcpy.RasterToNumPyArray(
                            ahn_clip,
                            nodata_to_value=NODATA
                        ).astype(np.float32)

                        ahn_arr[ahn_arr == NODATA] = np.nan

                    mask_arr = ~np.isnan(ahn_arr)
                
                except RuntimeError as e:
                
                    if "pixel block exceeds the maximum size allowed" in str(e).lower():
                        print(f"--- Raster te groot voor NumPy-conversie: "
                         f"{ws} / {polder} --> overslaan ---"
                    )
                    
                    continue
                
                    raise
                
#                 print("9 RasterToNumPyArray GEREED")
                # --------------------------------------------------
                # Landuse
                # --------------------------------------------------
                lu = arcpy.Raster(lu_raster)
                lu_clip = arcpy.sa.ExtractByMask(lu, layer)

                lu_arr = arcpy.RasterToNumPyArray(
                    lu_clip,
                    nodata_to_value=-1
                ).astype("int16")

                # --------------------------------------------------
                # peilgebied raster
                # --------------------------------------------------
                # Zorg dat peilgebied numeriek is
                if fld_peilgebied not in [f.name for f in arcpy.ListFields(fc) if f.type in ("Integer", "SmallInteger")]:
                    if "pg_id" not in [f.name for f in arcpy.ListFields(fc)]:
                        arcpy.AddField_management(fc, "pg_id", "LONG")
                        arcpy.CalculateField_management(fc, "pg_id", "!OBJECTID!", "PYTHON3")
                    fld_pg = "pg_id"
                    df["pg_id"] = df.index + 1  # fallback mapping
                else:
                    fld_pg = fld_peilgebied

                pg_raster = os.path.join(arcpy.env.scratchGDB, f"pg_{polder_norm}")

                arcpy.env.snapRaster = ahn_clip
                arcpy.env.extent = ahn_clip
                arcpy.env.cellSize = ahn_clip

                pg_raster = arcpy.conversion.PolygonToRaster(
                    fc,
                    fld_pg,
                    pg_raster,
                    cell_assignment="MAXIMUM_AREA",
                    priority_field=fld_pg,
                    cellsize=cellsize
                )

                pg_raster = arcpy.Raster(pg_raster)

                pg_arr = arcpy.RasterToNumPyArray(pg_raster)
                                
                arcpy.Delete_management(layer)

                # --------------------------------------------------
                # Berekeningen
                # --------------------------------------------------
                print(f"Start loop met {len(df)} peilgebied(en)")
                
                for _, row in df.iterrows():
                    
#                     print(f"Verwerken peilgebied: {row['peilgebied']}")
                    
                    pg_val = row[fld_peilgebied] if fld_pg == fld_peilgebied else row["pg_id"]
                    mask_pg = (pg_arr == pg_val)
                    
                    elevaties_pg = ahn_arr[mask_pg]
                    elevaties_pg = elevaties_pg[~np.isnan(elevaties_pg)] 
                    
                    lu_aantal_cellen = [0,0,0,0,0,0]
                    lu_dieptesommen = [0,0,0,0,0,0]
                    
                    wls = np.arange(row["maxpeil"],row["maxpeil"] + hoogte_x + interval,interval)
                    
                    for lu in [0, 1, 2, 3, 4, 5]:
                        lu_pg = ahn_arr[mask_pg & (lu_arr == lu)]
                        lu_pg = lu_pg[~np.isnan(lu_pg)]
                        
                        lu_aantal_cellen[lu], lu_dieptesommen[lu] = hypsometrie(
                            lu_pg,
                            wls
                        )
                    
                    aantal_cellen, dieptesommen = hypsometrie(elevaties_pg, wls)

                    for i, WL in enumerate(wls):
                        resultaten.append([
                            row["waterschap"],
                            row["polder"],
                            row["ws_id"],
                            row["opp_pldr"],
                            row["peilgebied"],
                            row["opp_pg"],
                            peil_type,
                            row["maalpand"],
                            WL,
                            
                            dieptesommen[i] * cellarea,
                            aantal_cellen[i] * cellarea,
                            
                            lu_aantal_cellen[0][i] * cellarea, # water
                            lu_aantal_cellen[1][i] * cellarea, # grasland
                            lu_aantal_cellen[2][i] * cellarea, # akker
                            lu_aantal_cellen[3][i] * cellarea, # hoogwaardig land -en tuinbouw 
                            lu_aantal_cellen[4][i] * cellarea, # bebouwing binnen bebouwde kom
                            lu_aantal_cellen[5][i] * cellarea, # bebouwing buiten bebouwde kom
                            
                            lu_dieptesommen[0][i] * cellarea, # water 
                            lu_dieptesommen[1][i] * cellarea, # grasland
                            lu_dieptesommen[2][i] * cellarea, # akker
                            lu_dieptesommen[3][i] * cellarea, # hoogwaarding land -en tuinbouw
                            lu_dieptesommen[4][i] * cellarea, # bebouwing binnen bebouwde kom
                            lu_dieptesommen[5][i] * cellarea, # bebouwing buiten bebouwde kom
                        ])
                             
                print(f"Berekening succesvol afgerond voor: "
                f"'{polder}' met {len(df)} peilgebieden.")

                # tussentijds opslaan
                pd.DataFrame(
                    resultaten,
                    columns=[
                        "waterschap", "polder", "WS_ID", "opp_pldr_m2",
                        "peilgebied", "opp_pg_m2", "peil_type",
                        "maalpand", "WL",
                        "volume_m3", "inundatie_m2",
                        # oppervlaktes
                        "water_m2", "grasland_m2", "akker_m2",
                        "tuinbouw_m2", "bebouwing_binnen_m2", "bebouwing_buiten_m2",
                        # volumes 
                        "water_m3", "grasland_m3", "akker_m3",
                        "tuinbouw_m3", "bebouwing_binnen_m3", "bebouwing_buiten_m3"
                    ]
                ).to_csv(output_csv, index=False)

    return pd.DataFrame(
        resultaten,
        columns=[
            "waterschap",
            "polder",
            "WS_ID",
            "opp_pldr_m2",
            "peilgebied",
            "opp_pg_m2",
            "peil_type",
            "maalpand",
            "WL",
            "volume_m3",
            "inundatie_m2",
            "water_m2",
            "grasland_m2",
            "akker_m2",
            "tuinbouw_m2",   
            "bebouwing_binnen_m2",
            "bebouwing_buiten_m2",
            "water_m3",
            "grasland_m3",
            "akker_m3",
            "tuinbouw_m3",
            "bebouwing_binnen_m3",
            "bebouwing_buiten_m3"
        ]
    )


### 1.1 Invoer: Bereken volume en inundatie per x cm peilstijging

**Doel**
Bepalen van inundatievolumes, inundatieoppervlaktes en de verdeling hiervan over verschillende landgebruikstypen voor alle peilgebieden binnen geslecteerde polders.

**Configuratie**
- Geef root_folder aan waar rekengebied mappen zich bevinden, eventueel specificeren tot één waterschap én ook nog polder
- Kies te gebruiken peiltype (mits deze te vinden is in rekengebied attributetable), bijv. zomerpeil (e.g. Peil_zo)
- Instellen van waterstandstap (interval)
- Instellen van maximale waterstandsverhoging boven uitgangspeil
- Koppeling van benodigde velden voor peilgebieden, peil en oppervlaktes.

**Voorbeeldconfiguratie**
- Waterschap: HHNK
- Polder: Eilandspolder
- Peiltype: zomerpeil (Peil_zo)
- Waterstandsinterval: 0,01 m (1 cm)
- Maximale waterstandsverhoging: 1,0 m boven het uitgangspeil

**Output**

- DataFrame met per peilgebied en waterstand:
    - inundatievolume (m³)
    - inundatieoppervlak (m²)
    - waterstand (m NAP)
    - oppervlakte van peilgebied en polder
    - type maalpand
- Uitsplitsing van inundatieoppervlaktes en volumes naar:
    - water
    - grasland
    - akkerbouw
    - hoogwaardig land- en tuinbouw
    - bebouwing
    - CSV-bestand met tussentijdse resultaten voor verdere analyse of controle


In [5]:
root_folder = r"D:\04_results"

df = bereken_volumes_fast(
    root_folder=root_folder,
    fld_waterschap="Waterschap",
    fld_polder="Naam_1",
    fld_peilgebied="CODE",
    fld_maxpeil="Peil_zo",
    fld_maalpand="Type",
    fld_op_pg="OPP_PG",
    fld_op_pldr="OPP_PLDR",

    peil_type="zo",
    interval=0.01,
    hoogte_x=3, 

    # optioneel:
    select_waterschap="HDSR",
    select_polder=["Amerongerwetering"]
)

df

# checken welke polders er in je dataframe zitten
# for p in df["polder"].unique():
#     print(p)


Waterschappen:   0%|          | 0/6 [00:00<?, ?it/s]

Polders hdsr:   0%|          | 0/99 [00:00<?, ?it/s]

--- hdsr / de_koekoek ---
--- beide rasters gevonden ---
--- start berekening ---


RuntimeError: The pixel block exceeds the maximum size allowed.

### Tussentijdse stap: sla gegenereerde dataset op in drievoud (.pkl, .txt & .csv)

**Doel**
Opslaan van de berekende inundatieresultaten in verschillende bestandsformaten voor verdere analyse, uitwisseling en hergebruik. 


**Werking**
- Aanmaken van de uitvoermap indien deze nog niet bestaat
- Wegschrijven van de resultaatdataset naar meerdere bestandsformaten

**Output**
- PKL-bestand (.pkl): Python-native opslagformaat voor snel hergebruik in analysescripts
- TXT-bestand (.txt): tab-gescheiden tekstbestand, geschikt voor gebruik in ArcGIS en andere GIS-software
- CSV-bestand (.csv): algemeen uitwisselingsformaat voor Excel, Power BI en andere analysetools 

**Resultaat**:
De volledige resultatentabel wordt opgeslagen in meerdere formaten, zodat deze zowel binnen Python als in GIS- en rapportagetools eenvoudig kan worden gebruikt.

In [52]:
import os
import pandas as pd

output_folder = r"D:\04_results\hdsr_results"
os.makedirs(output_folder, exist_ok=True)

# Basisnaam
base_name = "df_ameronger"

# Kopie maken zodat origineel intact blijft
df_export = df.copy()

# Waterstand afronden
df_export["WL"] = df_export["WL"].round(2)

# Oppervlaktes en volumes afronden
numerieke_kolommen = [
    "volume_m3",
    "inundatie_m2",
    "water_m2",
    "grasland_m2",
    "akker_m2",
    "tuinbouw_m2",
    "bebouwing_binnen_m2",
    "bebouwing_buiten_m2",
    "water_m3",
    "grasland_m3",
    "akker_m3",
    "tuinbouw_m3",
    "bebouwing_binnen_m3",
    "bebouwing_buiten_m3"
]

# Alleen kolommen afronden die daadwerkelijk bestaan
bestaande_kolommen = [
    col for col in numerieke_kolommen
    if col in df_export.columns
]

df_export[bestaande_kolommen] = (
    df_export[bestaande_kolommen]
    .round(1)
)

# Pickle
df_export.to_pickle(
    os.path.join(output_folder, f"{base_name}.pkl")
)

# TXT (tab-separated)
df_export.to_csv(
    os.path.join(output_folder, f"{base_name}.txt"),
    sep="\t",
    index=False
)

df_export.to_csv(
    os.path.join(output_folder, f"{base_name}.csv"),
    sep=";",
    index=False,
    float_format="%.2f"
)

Bestanden opgeslagen in: D:\04_results\27072026


# Deel 2 - Aggregeer dataset naar polderniveau

### 2.1 Inladen van resultaten in dataframe

Voordat dataset naar polderniveau geaggregeerd wordt, moet de resulterende dataset uit deel 1 opnieuw worden ingeladen. Dat kan hieronder:

In [7]:
df_hdsr = pd.read_csv(
    r"D:\04_results\hdsr_results\df_rvw_hdsr.csv", sep=";")
    
# df_hdsr = pd.read_csv(
#     r"D:\04_results\hdsr_results\df_rvw_hdsr.csv")
    
# df_hhr = pd.read_csv(
#     r"D:\04_results\hhr_results\df_rvw_hhr.csv")

# laat begin en eind of gehele dataframe zien
# pd.set_option("display.max_rows", None)
pd.reset_option("display.max_rows")

df_hdsr

### 2.2 Aggregeer dataframe naar polderniveau

In dit deel wordt de dataframe naar polderniveau geaggregeerd.


### 2.2 Achtergrondfunctie: aggregeer dataframe naar polderniveau

**Doel**
- Samenvoegen van inundatieresultaten op peilgebiedniveau tot één consistente waterstandsreeks per polder.

**Werking**
- Groeperen van resultaten per waterschap en polder.
- Samenvoegen van inundatievolumes, inundatieoppervlaktes en landgebruiksstatistieken van alle peilgebieden.
- Vastleggen van de geaggregeerde resultaten per waterstand.

**Output**
- Inundatievolume per polder en waterstand.
- Inundatieoppervlak per polder en waterstand.
- Uitsplitsing naar landgebruikstype.


In [8]:
def aggregate_to_polder_level(df):
    
    """
    Aggregeert inundatieresultaten van peilgebiedniveau naar polderniveau.

    Voor iedere polder wordt per waterstand (WL) het totale inundatievolume,
    inundatieoppervlak en de verdeling naar landgebruik bepaald. Hierbij
    worden de resultaten van alle peilgebieden binnen dezelfde polder
    samengevoegd.

    Returns
    -------
    pandas.DataFrame

    DataFrame met per polder en waterstand:

    - waterschap
    - polder
    - waterstand (WL)
    - oppervlakte polder
    - totale oppervlakte peilgebieden
    - inundatievolume (m³)
    - inundatieoppervlak (m²)

    Daarnaast worden voor de volgende landgebruiksklassen zowel
    oppervlaktes als volumes bepaald:

    - water
    - grasland
    - akker
    - hoogwaardig land- en tuinbouw (afgekort: tuinbouw)
    - bebouwing

    Werkwijze
    ---------
    1. Groepeer de invoergegevens per waterschap en polder.
    2. Bepaal per polder het bereik van waterstanden.
    3. Doorloop iedere waterstand binnen dit bereik.
    4. Selecteer per peilgebied de laatst beschikbare berekening die
       kleiner dan of gelijk is aan de huidige waterstand.
    5. Tel volumes, inundatieoppervlaktes en landgebruiksstatistieken
       van alle peilgebieden binnen de polder bij elkaar op.
    6. Sla de geaggregeerde resultaten op in een nieuwe DataFrame.
    7. Retourneer de resultaten op polderniveau.

    Opmerkingen
    -----------
    Omdat peilgebieden verschillende uitgangspeilen kunnen hebben,
    wordt per waterstand steeds de laatst beschikbare berekening van
    ieder peilgebied gebruikt. Hierdoor ontstaat een consistente
    waterstandsreeks op polderniveau.
    """
    import numpy as np
    df = df.copy()

    df["waterschap"] = df["waterschap"].astype(str).str.strip()
    df["polder"] = df["polder"].astype(str).str.strip()
    df["WL"] = df["WL"].astype(float)

    result_rows = []

    # per polder
    for (ws, polder), df_pol in df.groupby(["waterschap", "polder"]):

        # globale WL range
        wl_min = df_pol["WL"].min()
        wl_max = df_pol["WL"].max()

        WL_range = np.arange(wl_min, wl_max + 0.001, 0.02)

        #per WL
        for WL in WL_range:

            volume_sum = 0
            inundatie_sum = 0
            water_m2_sum = 0
            gras_m2_sum = 0
            akker_m2_sum = 0
            tuinbouw_m2_sum = 0
            bebouwing_binnen_m2_sum = 0
            bebouwing_buiten_m2_sum = 0
            water_m3_sum = 0
            gras_m3_sum = 0
            akker_m3_sum = 0
            tuinbouw_m3_sum = 0
            bebouwing_binnen_m3_sum = 0
            bebouwing_buiten_m3_sum = 0
            opp_pg_sum = 0

            # loop over peilgebieden
            for pg, df_pg in df_pol.groupby("peilgebied"):

                # pak alle waarden ≤ huidige WL
                df_pg_valid = df_pg[df_pg["WL"] <= WL]

                if df_pg_valid.empty:
                    continue

                # neem laatste (hoogste WL onder huidige WL)
                row = df_pg_valid.iloc[-1]

                volume_sum += row["volume_m3"]
                inundatie_sum += row["inundatie_m2"]
                water_m2_sum += row["water_m2"]
                gras_m2_sum += row["grasland_m2"]
                akker_m2_sum += row["akker_m2"]
                tuinbouw_m2_sum += row["tuinbouw_m2"]
                bebouwing_binnen_m2_sum += row["bebouwing_binnen_m2"]
                bebouwing_buiten_m2_sum += row["bebouwing_buiten_m2"]
                water_m3_sum += row["water_m3"]
                gras_m3_sum += row["grasland_m3"]
                akker_m3_sum += row["akker_m3"]
                tuinbouw_m3_sum += row["tuinbouw_m3"]
                bebouwing_binnen_m3_sum += row["bebouwing_binnen_m3"]
                bebouwing_buiten_m3_sum += row["bebouwing_buiten_m3"]
                opp_pg_sum += row["opp_pg_m2"]

            result_rows.append([
                ws,
                polder,
                WL,
                df_pol["opp_pldr_m2"].iloc[0],
                opp_pg_sum,
                volume_sum,
                inundatie_sum,
                water_m2_sum,
                gras_m2_sum,
                akker_m2_sum,
                tuinbouw_m2_sum,
                bebouwing_binnen_m2_sum,
                bebouwing_buiten_m2_sum,
                water_m3_sum,
                gras_m3_sum,
                akker_m3_sum,
                tuinbouw_m3_sum,
                bebouwing_binnen_m3_sum,
                bebouwing_buiten_m3_sum
            ])

    df_out = pd.DataFrame(result_rows, columns=[
        "waterschap",
        "polder",
        "WL",
        "opp_pldr_m2",
        "opp_pg_m2",
        "volume_m3",
        "inundatie_m2",
        "water_m2",
        "grasland_m2",
        "akker_m2",
        "tuinbouw_m2",
        "bebouwing_binnen_m2",
        "bebouwing_buiten_m2",
        "water_m3",
        "grasland_m3",
        "akker_m3",
        "tuinbouw_m3",
        "bebouwing_binnen_m3",
        "bebouwing_buiten_m3"
    ])

    return df_out



### 2.2 Invoer: aggregeer dataframe naar polderniveau

Roep aggregate_to_polder_level functie op met genoemde dataframe in deel 2.1 en laat uitvoeren.


In [33]:
df_hdsr_agg = aggregate_to_polder_level(df_hdsr)

df_hdsr_agg.to_csv(
    os.path.join(r"D:\04_results\hdsr_results\df_rvw_hdsr_agg.csv"),
    sep=";",
    index=False,
    float_format="%.2f")

# Deel 3 - Bereken RvW o.b.v. Ontwerpbui T

### 3.1 Genereer output .txt file + inundatieraster per polder 

**Doel**  
Deze stap vertaalt de eerder berekende volume-waterstandrelaties op polderniveau naar inundatiekaarten voor specifieke neerslagscenario's. Hiermee kan worden bepaald welke waterstand nodig is om een bepaalde hoeveelheid neerslag binnen een polder te bergen en welke delen van de polder daarbij inunderen.

**Achtergrond**
De inundatieberekeningen op polderniveau leveren een relatie tussen waterstand, inundatieoppervlak en bergingsvolume. Op basis van een opgegeven neerslaghoeveelheid wordt eerst het benodigde bergingsvolume bepaald:

```text
Bergingsvolume = Neerslaghoogte × Polderoppervlak
```

Vervolgens wordt binnen de beschikbare volume-waterstandrelatie gezocht naar de waterstand waarvan het berekende inundatievolume het dichtst bij dit benodigde bergingsvolume ligt.

**Werking**
- Bepalen van het benodigde bergingsvolume op basis van neerslag en polderoppervlak.
- Selecteren van de best passende waterstand uit de berekende volume-waterstandrelatie.
- Berekenen van de inundatiediepte door de geselecteerde waterstand te vergelijken met de maaiveldhoogte.


**Output**
Per polder worden de volgende bestanden gegenereerd:

- TXT-bestand met de geselecteerde waterstand en bijbehorende inundatiekenmerken.
- GeoTIFF-bestand met de berekende inundatiedieptes.

Daarnaast wordt per polder vastgelegd:

- waterschap
- polder
- polderoppervlak
- neerslagvolume
- geselecteerde waterstand
- inundatievolume


### 3.1 Achtergrondfunctie: Genereer output .txt file + inundatieraster per polder
Desbetreffende functie van boventstaande omschrijving

In [19]:
def find_closest_volume(df, waterschap, polder, target_volume):
    
    """
    Zoekt binnen een polder de waterstand waarvan het berekende
    inundatievolume het dichtst bij een opgegeven doelvolume ligt.
    Retourneert de volledige rij met resultaten.
    """
    df_pol = df[(df["waterschap"] == waterschap) &
                (df["polder"] == polder)].copy()   # ← copy() voorkomt ook de warning

    df_pol.loc[:, "abs_diff"] = (df_pol["volume_m3"] - target_volume).abs()

    return df_pol.loc[df_pol["abs_diff"].idxmin()]

def format_wl(value):
    
    """
    Formatteert een waterstand voor gebruik in bestandsnamen (is nu niet in gebruik maar optioneel).

    Voorbeelden:
    - -0,37   -> m0p37
    - -0.375  -> m0p38
    - 12      -> 12p00
    """
    s = str(value).strip()

    # Komma → punt
    s = s.replace(",", ".")

    # Naar float en afronden
    try:
        f = float(s)
    except ValueError:
        raise ValueError(f"Kan WL niet formatteren: {value}")

    f = round(f, 2)

    # Negatief teken veilig maken
    prefix = "m" if f < 0 else ""
    f = abs(f)

    # Punt vervangen door 'p'
    s = f"{f:.2f}".replace(".", "p")

    return f"{prefix}{s}"

from arcpy.sa import CreateConstantRaster, Raster

def process_polders_from_df_with_shapefile(
    df_per_polder,
    shapefile,
    polderpath_root,
    rain_mm,
 
    # ---- dataframe field mapping
    df_fld_waterschap,
    df_fld_polder,
 
    # ---- shapefile field mapping
    shp_fld_waterschap,
    shp_fld_polder,
    shp_fld_op_pldr,
 
    peil_type="zo",
    output_folder_name="resultaat"
):
    
    """
    Verwerkt inundatieresultaten op polderniveau voor een opgegeven
    neerslaggebeurtenis en genereert inundatiekaarten per polder.

    Voor iedere polder wordt op basis van het polderoppervlak en een
    opgegeven neerslaghoeveelheid een doelvolume bepaald. Vervolgens wordt
    uit de berekende volume-waterstandrelatie de waterstand geselecteerd
    waarvan het volume het dichtst bij dit doelvolume ligt.

    Per polder wordt een inundatieraster gemaakt waarin de waterdiepte
    zichtbaar is die nodig is om het berekende neerslagvolume te bergen.

    Returns
    -------
    Per polder worden de volgende bestanden aangemaakt:

    - TXT-bestand met de geselecteerde waterstand en bijbehorende
      inundatiekenmerken.
    - GeoTIFF-raster met de berekende inundatiediepte.

    Daarnaast wordt per polder bepaald:

    - waterschap
    - polder
    - polderoppervlak
    - neerslagvolume
    - geselecteerde waterstand (WL)
    - inundatievolume

    Werkwijze
    ---------
    1. Controleer of alle vereiste velden aanwezig zijn.
    2. Groepeer de invoergegevens per waterschap en polder.
    3. Lees het polderoppervlak uit de shapefile.
    4. Bereken het doelvolume op basis van de opgegeven neerslag:

       volume = neerslag (m) × polderoppervlak (m²)

    5. Zoek de waterstand waarvan het inundatievolume het dichtst
       bij het doelvolume ligt.
    6. Sla de geselecteerde resultaten op als TXT-bestand.
    7. Lees het peilraster van de polder.
    8. Bereken de inundatiediepte als:

       inundatie = waterstand − maaiveldhoogte

    9. Verwijder negatieve waarden zodat alleen inundatie wordt
       weergegeven.
    10. Sla het inundatieraster op als GeoTIFF.

    Opmerkingen
    -----------
    De functie gebruikt eerder berekende volume-waterstandrelaties op
    polderniveau. Hierdoor kan voor verschillende neerslagscenario's
    snel een inundatiekaart worden gegenereerd zonder de volledige
    volumeberekening opnieuw uit te voeren.
    """

    arcpy.CheckOutExtension("Spatial")
 
    # --------------------------------------------------
    # Validate dataframe schema
    for fld in [df_fld_waterschap, df_fld_polder]:
        if fld not in df_per_polder.columns:
            raise KeyError(
                f"Kolom '{fld}' ontbreekt in df_per_polder"
            )
 
    # --------------------------------------------------
    # Validate shapefile schema
    shp_fields = [f.name for f in arcpy.ListFields(shapefile)]
    for fld in [shp_fld_waterschap, shp_fld_polder, shp_fld_op_pldr]:
        if fld not in shp_fields:
            raise KeyError(
                f"Veld '{fld}' ontbreekt in shapefile"
            )
 
    # --------------------------------------------------
    # Loop over polders (bron van waarheid = df + folderstructuur)
    grouped = df_per_polder.groupby(
        [df_fld_waterschap, df_fld_polder]
    )
 
    for (waterschap, polder), df_pol in grouped:
 
        print(f"--- Verwerk {waterschap} / {polder} ---")
 
        ws_norm = maak_veilige_naam(waterschap, target="filesystem")
        pol_norm = maak_veilige_naam(polder, target="filesystem")
        
        
        polder_path = os.path.join(
            polderpath_root,
            ws_norm,
            pol_norm
        )

        # --------------------------------------------------
        # 🔗 Lees + sommeer oppervlak uit shapefile
        waterschap_sql = str(waterschap).replace("'", "''")
        polder_sql = str(polder).replace("'", "''")
        
        where = (
            f"{shp_fld_waterschap} = '{waterschap_sql}' AND "
            f"{shp_fld_polder} = '{polder_sql}'"
        )
 
        opp_pldr_m2 = 0.0
        count = 0
 
        with arcpy.da.SearchCursor(
            shapefile,
            [shp_fld_op_pldr],
            where_clause=where
        ) as cursor:
            for (opp,) in cursor:
                if opp is not None:
                    opp_pldr_m2 += float(opp)
                    count += 1
 
        if count == 0:
            raise ValueError(
                f"Geen oppervlak gevonden in shapefile voor "
                f"{waterschap} / {polder}"
            )
 
        if opp_pldr_m2 <= 0:
            raise ValueError(
                f"Ongeldig polderoppervlak voor "
                f"{waterschap} / {polder}: {opp_pldr_m2}"
            )
 
        print(
            f"    • oppervlak uit shapefile: "
            f"{count} deelgebieden, totaal = {int(opp_pldr_m2)} m²"
        )
 
        # --------------------------------------------------
        # Target volume from rainfall
        target_volume = (rain_mm / 1000.0) * opp_pldr_m2
 
        result = find_closest_volume(
            df_pol,
            waterschap,
            polder,
            target_volume
        )
 
        WL_raw = result["WL"]
        WL_fmt = format_wl(WL_raw)
        
        # --------------------------------------------------
        # Laagste peil uit rekengebied ophalen

        rekengebied_fc = os.path.join(
            polder_path,
            f"{pol_norm}.gdb",
            f"rekengebied_{pol_norm}"
        )

        peilveld = "Peil_zo" if peil_type.lower() == "zo" else "Peil_wi"

        if not arcpy.Exists(rekengebied_fc):
            raise FileNotFoundError(
                f"Rekengebied ontbreekt: {rekengebied_fc}"
            )

        hoofdvak_peil = None

        with arcpy.da.SearchCursor(
            rekengebied_fc,
            ["Type", peilveld]
        ) as cursor:

            for row in cursor:
                if (
                    row[0] is not None
                    and str(row[0]).strip().lower() == "hoofdvak"
                    and row[1] is not None
                ):
                    hoofdvak_peil = float(row[1])
                    break

        if hoofdvak_peil is None:

            print(
                f"Waarschuwing: geen Hoofdvak gevonden "
                f"voor {waterschap} / {polder}"
            )

            result["hoofdvak_peil"] = np.nan
            result["peilstijging_m"] = np.nan

        else:

            peilstijging = WL_raw - hoofdvak_peil

            result["hoofdvak_peil"] = hoofdvak_peil
            result["peilstijging_m"] = peilstijging
 
        # --------------------------------------------------
        # Paths
 
        out_dir = os.path.join(polder_path, output_folder_name)
        os.makedirs(out_dir, exist_ok=True)
 
        # --------------------------------------------------
        # Write TXT
        rain_tag = f"bui{int(rain_mm)}mm"

        txt_path = os.path.join(
            out_dir,
            f"{ws_norm}_{pol_norm}_{rain_tag}.txt"
        )
 
        result.to_frame().T.to_csv(
            txt_path,
            sep="\t",
            index=False
        )
 
        # --------------------------------------------------
        # Raster calculation: WL – peil
        peil_raster = os.path.join(
            polder_path,
            f"{ws_norm}_{pol_norm}_peil_{peil_type}.tif"
        )
         
        if not arcpy.Exists(peil_raster):
            raise FileNotFoundError(
                f"Peilraster ontbreekt: {peil_raster}"
            )
 
        with arcpy.EnvManager(
            extent=peil_raster,
            cellSize=peil_raster,
            snapRaster=peil_raster
        ):
            inundatie = (
                arcpy.sa.Float(WL_raw)
                - (arcpy.sa.Raster(peil_raster) / 1000)
            )
 
            inundatie = SetNull(inundatie <= 0, inundatie)
 
            out_raster = os.path.join(
                out_dir,
                f"{ws_norm}_{pol_norm}_{rain_tag}.tif"
            )
 
            inundatie.save(out_raster)
 
        print(
            f"  ✓ WL={WL_raw}, "
            f"neerslag={rain_mm} mm, "
            f"volume≈{int(target_volume)} m³"
        )


### 3.1 Invoer: genereer output .txt file + inundatieraster per polder 
Voor het genereren van inundatiekaarten moeten de volgende gegevens worden opgegeven:
- **df_per_polder**: DataFrame met de geaggregeerde volume-waterstandrelaties op polderniveau.
- **shapefile**: Feature class of shapefile met de poldergeometrieën en oppervlakten.
- **polderpath_root**: Hoofdmap waarin de polderresultaten en rasters zijn opgeslagen.
- **rain_mm**: Neerslaghoeveelheid (mm) waarvoor de inundatiekaart wordt berekend.
- **df_fld_waterschap**: Kolomnaam in de DataFrame met de waterschapsnaam.
- **df_fld_polder**: Kolomnaam in de DataFrame met de poldernaam.
- **shp_fld_waterschap**: Veldnaam in de shapefile met de waterschapsnaam.
- **shp_fld_polder**: Veldnaam in de shapefile met de poldernaam.
- **shp_fld_op_pldr**: Veldnaam met de oppervlakte van de polderdelen. Indien een polder uit meerdere polygonen bestaat worden deze oppervlakten automatisch gesommeerd.
- **peil_type**: Type peilraster dat wordt gebruikt, bijvoorbeeld zomerpeil (`zo`) of winterpeil (`wi`).
 
**Voorbeeldconfiguratie**
```text
Neerslagscenario : 92 mm
Waterschapsveld : Waterschap
Polderveld : Naam_1
Oppervlakteveld : Shape_Area
Peiltype : zomerpeil (zo)
Resultaatlocatie: D:\04_results
```


In [20]:
process_polders_from_df_with_shapefile(
    df_per_polder=df_agv_agg,
    shapefile=r"C:\Users\Senden02\OneDrive - Waternet Amsterdam\Documenten\ArcGIS\Projects\RvW-DPCH\rvw_tool\01_src\03_data_processing\ahn_merger\ahn_merger.gdb\afvoergebieden_compleet_agv", #vul hier het rekengebied in
    polderpath_root=r"D:\04_results",
    rain_mm=92,

    # ---- dataframe field mapping
    df_fld_waterschap="waterschap",
    df_fld_polder="polder",

    # ---- shapefile field mapping
    shp_fld_waterschap="Waterschap",
    shp_fld_polder="Naam_1",
    shp_fld_op_pldr="Shape_Area", #De area van de peilvakken [wordt gesommeerd]

    peil_type="zo"
)

--- Verwerk AGV / 's-Gravelandsche Polder ---
    • oppervlak uit shapefile: 1 deelgebieden, totaal = 8480336 m²
  ✓ WL=0.48, neerslag=92 mm, volume≈780190 m³
--- Verwerk AGV / 's-Gravelandsche vaartboezem ---
    • oppervlak uit shapefile: 1 deelgebieden, totaal = 7363107 m²
Waarschuwing: geen Hoofdvak gevonden voor AGV / 's-Gravelandsche vaartboezem
  ✓ WL=0.16, neerslag=92 mm, volume≈677405 m³
--- Verwerk AGV / Aetsveldse Polder Oost ---
    • oppervlak uit shapefile: 1 deelgebieden, totaal = 8547762 m²
  ✓ WL=-1.44, neerslag=92 mm, volume≈786394 m³
--- Verwerk AGV / Aetsveldse Polder west ---
    • oppervlak uit shapefile: 1 deelgebieden, totaal = 2777844 m²
  ✓ WL=-1.45, neerslag=92 mm, volume≈255561 m³
--- Verwerk AGV / Aetsveldse Polder west (Driemond) ---
    • oppervlak uit shapefile: 1 deelgebieden, totaal = 67416 m²
Waarschuwing: geen Hoofdvak gevonden voor AGV / Aetsveldse Polder west (Driemond)
  ✓ WL=-0.75, neerslag=92 mm, volume≈6202 m³
--- Verwerk AGV / Atekpolder ---
 

  ✓ WL=-2.21, neerslag=92 mm, volume≈279795 m³
--- Verwerk AGV / Polder Holland en Sticht west ---
    • oppervlak uit shapefile: 1 deelgebieden, totaal = 2248935 m²
  ✓ WL=-1.64, neerslag=92 mm, volume≈206902 m³
--- Verwerk AGV / Polder Kortenhoef ---
    • oppervlak uit shapefile: 1 deelgebieden, totaal = 17238904 m²
  ✓ WL=-0.98, neerslag=92 mm, volume≈1585979 m³
--- Verwerk AGV / Polder Maarsseveen-Westbroek ---
    • oppervlak uit shapefile: 1 deelgebieden, totaal = 18927535 m²
  ✓ WL=-0.79, neerslag=92 mm, volume≈1741333 m³
--- Verwerk AGV / Polder Mijnden ---
    • oppervlak uit shapefile: 1 deelgebieden, totaal = 3087800 m²
  ✓ WL=-1.06, neerslag=92 mm, volume≈284077 m³
--- Verwerk AGV / Polder Nijenrode ---
    • oppervlak uit shapefile: 1 deelgebieden, totaal = 3075434 m²
  ✓ WL=-0.83, neerslag=92 mm, volume≈282939 m³
--- Verwerk AGV / Polder Oukoop en Polder Groot Wilnis-Vinkeveen (oo ---
    • oppervlak uit shapefile: 1 deelgebieden, totaal = 9693262 m²
  ✓ WL=-1.95, neersl

### 3.2 Genereer landgebruik tabel 

**Doel**

In deze tussenstap wordt een .csv bestand gegenereert waarin per polder het oppervlakte van elk landgebruik aanwezig is. Dit .csv bestand wordt in de volgende stap gebruikt om te kunnen analyseren wat de hoeveelheid RvW in m² is per landgebruik ten opzichte van het totaal aanwezige oppervlak in m² van dat landgebruik type.

In [21]:
def maak_landgebruik_tabel(
    df_per_polder,
    root_folder
):

    resultaten = []

    grouped = df_per_polder.groupby(
        ["waterschap", "polder"]
    )

    for (waterschap, polder), _ in grouped:

        ws_norm = maak_veilige_naam(
            waterschap,
            target="filesystem"
        )

        pol_norm = maak_veilige_naam(
            polder,
            target="filesystem"
        )

        lu_raster = os.path.join(
            root_folder,
            ws_norm,
            pol_norm,
            f"{ws_norm}_{pol_norm}_lu.tif"
        )

        if not arcpy.Exists(lu_raster):
            print(f"Niet gevonden: {lu_raster}")
            continue

        try:
            arcpy.management.BuildRasterAttributeTable(
                lu_raster,
                "Overwrite"
            )
        except:
            pass

        cellsize = float(
            arcpy.management.GetRasterProperties(
                lu_raster,
                "CELLSIZEX"
            ).getOutput(0).replace(",", ".")
        )

        cel_oppervlak = cellsize * cellsize

        lu_counts = {}

        with arcpy.da.SearchCursor(
            lu_raster,
            ["VALUE", "COUNT"]
        ) as cursor:

            for value, count in cursor:
                lu_counts[int(value)] = int(count)

        resultaten.append({
            "waterschap": waterschap,
            "polder": polder,
            "water_tot_m2":
                lu_counts.get(0, 0) * cel_oppervlak,
            "grasland_tot_m2":
                lu_counts.get(1, 0) * cel_oppervlak,
            "akker_tot_m2":
                lu_counts.get(2, 0) * cel_oppervlak,
            "tuinbouw_tot_m2":
                lu_counts.get(3, 0) * cel_oppervlak,
            "bebouwing_binnen_tot_m2":
                lu_counts.get(4, 0) * cel_oppervlak,
            "bebouwing_buiten_tot_m2":
                lu_counts.get(5, 0) * cel_oppervlak,
        })

    return pd.DataFrame(resultaten)

In [11]:
root_folder = r"D:\04_results"

df_agv_agg = pd.read_csv(r"D:\04_results\agv_results\df_rvw_agv_agg.csv", sep=";")

df_landgebruik = maak_landgebruik_tabel(
    df_agv_agg,
    root_folder
)

df_landgebruik.to_csv(
    r"D:\04_results\agv_results\landgebruik_oppervlak.csv",
    sep=";",
    index=False
)

### 3.3 Inlezen van resultaten en berekenen van RvW 

**Doel**

Deze stap leest de resultaten van een neerslagscenario per polder in en berekent de RVW-indicator (Ruimte voor Water). De RVW-indicator geeft inzicht in de mate waarin economisch of maatschappelijk relevant landgebruik wordt getroffen door inundatie.

**Context**

Voor iedere polder is in een eerdere stap een resultatenbestand opgeslagen met inundatieoppervlaktes en inundatievolumes per landgebruikstype. Deze functie verzamelt deze resultaten en berekent daaruit een samenvattende RVW-indicator.

Bij de RVW-berekening wordt uitsluitend gekeken naar landgebruikstypen waarbij inundatie potentieel kan leiden tot schade of hinder:

- grasland
- akkerbouw
- hoogwaardig land- en tuinbouw
- bebouwing

Open water wordt hierbij buiten beschouwing gelaten.

**Output**

Per polder worden de volgende aanvullende indicatoren berekend:

- **rvw_m2**: totaal overstroomd oppervlak van grasland, akker, tuinbouw en bebouwing (m²)
- **rvw_m3**: totaal inundatievolume op grasland, akker, tuinbouw en bebouwing (m³)
- **rvw_%**: percentage van het totale polderoppervlak dat onder RVW valt

Daarnaast blijven alle eerder berekende inundatiekenmerken beschikbaar.


### 3.3 Achtergrondfunctie: Inlezen van resultaten en berekenen van RvW
Desbetreffende functie van boventstaande omschrijving

In [22]:
def lees_resultaten(
    df_per_polder,
    root_folder,
    rain_mm
):
    
    """
    Leest de resultaten van een neerslagscenario per polder in en
    berekent aanvullende RVW-indicatoren.

    Voor iedere polder wordt het eerder opgeslagen resultatenbestand
    (TXT) opgezocht en ingelezen. Vervolgens worden de oppervlaktes en
    volumes van de relevante landgebruikstypen samengevoegd tot één
    RVW-waarde.

    Returns
    -------
    pandas.DataFrame

    DataFrame met de ingelezen resultaten per polder, aangevuld met:

    - rvw_m2: totaal overstroomd oppervlak van grasland, akker,
      tuinbouw en bebouwing (m²)
    - rvw_m3: totaal inundatievolume op grasland, akker,
      tuinbouw en bebouwing (m³)
    - rvw_%: percentage van het polderoppervlak dat onder RVW valt

    Werkwijze
    ---------
    1. Groepeer de invoergegevens per waterschap en polder.
    2. Zoek voor iedere polder het resultatenbestand van het
       opgegeven neerslagscenario.
    3. Lees de resultatenbestanden in als pandas DataFrame.
    4. Voeg alle resultaten samen tot één totaaloverzicht.
    5. Bereken het totale RVW-oppervlak door de oppervlaktes van:
       - grasland
       - akker
       - tuinbouw
       - bebouwing
       op te tellen.
    6. Bereken het totale RVW-volume voor dezelfde landgebruikstypen.
    7. Bereken het RVW-percentage ten opzichte van het totale
       polderoppervlak.
    8. Retourneer de verrijkte DataFrame.

    Opmerkingen
    -----------
    Wateroppervlak wordt niet meegenomen in de RVW-berekening.
    De RVW-indicator richt zich uitsluitend op de landgebruikstypen
    waar inundatie tot schade of hinder kan leiden.
    """

    rain_tag = f"bui{int(rain_mm)}mm"

    dfs = []

    grouped = df_per_polder.groupby(
        ["waterschap", "polder"]
    )

    for (waterschap, polder), _ in grouped:

        ws_norm = maak_veilige_naam(
            waterschap,
            target="filesystem"
        )

        pol_norm = maak_veilige_naam(
            polder,
            target="filesystem"
        )

        result_dir = (
            Path(root_folder)
            / ws_norm
            / pol_norm
            / "resultaat"
        )

        matches = list(
            result_dir.glob(
                f"{ws_norm}_{pol_norm}_{rain_tag}*.txt"
            )
        )

        if not matches:
            print(f"Niet gevonden: {result_dir}")
            continue

        dfs.append(
            pd.read_csv(matches[0], sep="\t")
        )

    if not dfs:
        return pd.DataFrame()
        
        
    df_all = pd.concat(
            dfs,
            ignore_index=True
        )
    
    df_landgebruik = pd.read_csv(
    r"D:\04_results\agv_results\landgebruik_oppervlak.csv",
    sep=";"
    )

    df_all = df_all.merge(
        df_landgebruik,
        on=["waterschap", "polder"],
        how="left"
    )
    
    df_all = df_all.drop(
            columns=["peilgebied"],
            errors="ignore"
        )

    # ------------------------------
    # RVW-berekeningen

    df_all["rvw_m2"] = df_all[
        [
            "grasland_m2",
            "akker_m2",
            "tuinbouw_m2",
            "bebouwing_binnen_m2",
            "bebouwing_buiten_m2"
        ]
    ].sum(axis=1)

    df_all["rvw_m3"] = df_all[
        [
            "grasland_m3",
            "akker_m3",
            "tuinbouw_m3",
            "bebouwing_binnen_m3",
            "bebouwing_buiten_m3"
        ]
    ].sum(axis=1)

    df_all["rvw_m2_Proc"] = (
        df_all["rvw_m2"]
        / df_all["opp_pldr_m2"]
    ) * 100

    df_all["rvw_m3_Proc"] = (
        df_all["rvw_m3"]
        / df_all["volume_m3"]
    ) * 100

    df_all["sloot_Proc"] = (
        df_all["water_m3"]
        / df_all["volume_m3"]
    ) * 100

    df_all["water_m2_Proc"] = (
        df_all["water_m2"]
        / df_all["opp_pldr_m2"]
    ) * 100

    df_all["opp_pldr_ha"] = (
        df_all["opp_pldr_m2"] / 10000)

    df_all["gem_inun_m"] = ((df_all["grasland_m3"] + df_all["akker_m3"] + df_all["tuinbouw_m3"] + df_all["bebouwing_binnen_m3"] + df_all["bebouwing_buiten_m3"]) / 
                            (df_all["grasland_m2"] + df_all["akker_m2"] + df_all["tuinbouw_m2"] + df_all["bebouwing_binnen_m2"] + df_all["bebouwing_buiten_m2"])) * 100

    for lu in [
        "water",
        "grasland",
        "akker",
        "tuinbouw",
        "bebouwing_binnen",
        "bebouwing_buiten"
    ]:
        df_all[f"{lu}_Proc"] = np.where(
            df_all[f"{lu}_tot_m2"] > 0,
            df_all[f"{lu}_m2"] / df_all[f"{lu}_tot_m2"] * 100,
            0
        )

        # WL op 2 decimalen
    if "WL" in df_all.columns:
        df_all["WL"] = df_all["WL"].round(2)

    # Alle overige numerieke kolommen op 0 decimalen
    for col in df_all.select_dtypes(include="number").columns:
        if col not in ["WL", "hoofdvak_peil", "peilstijging_m", "gem_inun_m"]:
            df_all[col] = df_all[col].round(0)

    df_all["hoofdvak_peil"] = df_all["hoofdvak_peil"].round(2)
    df_all["peilstijging_m"] = df_all["peilstijging_m"].round(2)
    df_all["gem_inun_m"] = df_all["gem_inun_m"].round(2)
    
    column_order = [
    "waterschap",
    "polder",
    "WL",
    "hoofdvak_peil",
    "peilstijging_m",
    "opp_pldr_ha",
    "opp_pldr_m2",
    "volume_m3",
    "rvw_m2",
    "rvw_m3",
    "rvw_m2_Proc",
    "rvw_m3_Proc",
    "gem_inun_m",
    "water_m2",
    "water_m3",
    "water_m2_Proc",
    "sloot_Proc",
    ]

    # Add all remaining columns that are not explicitly listed
    column_order.extend(
        [col for col in df_all.columns if col not in column_order]
    )

    df_all = df_all[column_order]

    return df_all


### 3.2 Invoer: Inlezen van resultaten en berekenen van RvW


- **df_per_polder**: DataFrame met de geaggregeerde inundatieresultaten op polderniveau.
- **root_folder**: Hoofdmap waarin de resultaten per waterschap en polder zijn opgeslagen.
- **rain_mm**: Neerslagscenario waarvoor de resultaten moeten worden ingelezen.

**Resulteert in:**
De functie levert één verrijkte resultatentabel op waarin per polder zowel de inundatieresultaten als de afgeleide RVW-indicatoren beschikbaar zijn voor verdere analyse en rapportage.

In [23]:
# df_agv_polder  = pd.read_csv(r"D:\04_results\27072026\agv_results\df_rvw_agv_agg.csv", sep=";")

df_agv_rvw = lees_resultaten(
    df_per_polder=df_agv_agg,
    root_folder=r"D:\04_results",
    rain_mm=92
)

# df_agv_rvw.to_csv(
#     os.path.join(r"D:\04_results\agv_results\df_rvw_agv_agg.csv"),
#     sep=";",
#     index=False,
#     float_format="%.2f")

# pd.set_option("display.max_rows", None)
df_agv_rvw

,waterschap,polder,WL,opp_pldr_m2,opp_pg_m2,volume_m3,inundatie_m2,water_m2,grasland_m2,akker_m2,tuinbouw_m2,bebouwing_binnen_m2,bebouwing_buiten_m2,water_m3,grasland_m3,akker_m3,tuinbouw_m3,bebouwing_binnen_m3,bebouwing_buiten_m3,abs_diff,hoofdvak_peil,peilstijging_m,water_tot_m2,grasland_tot_m2,akker_tot_m2,tuinbouw_tot_m2,bebouwing_binnen_tot_m2,bebouwing_buiten_tot_m2,rvw_m2,rvw_m3,rvw_m2_Proc,rvw_m3_Proc,sloot_Proc,water_m2_Proc,opp_pldr_ha,gem_inun_m,water_Proc,grasland_Proc,akker_Proc,tuinbouw_Proc,bebouwing_binnen_Proc,bebouwing_buiten_Proc
0,AGV,'s-Gravelandsche Polder,0.48,8480336.0,6034040.0,784431.0,2630535.0,508358.0,1933006.0,20512.0,66540.0,92863.0,8152.0,306566.0,443609.0,3073.0,15918.0,14002.0,1030.0,4240.0,-0.20,0.68,577545.0,4750293.0,56843.0,152834.0,2816391.0,119300.0,2121073.0,477633.0,25.0,61.0,39.0,6.0,848.0,22.52,88.0,41.0,36.0,44.0,3.0,7.0
1,AGV,'s-Gravelandsche vaartboezem,0.16,7363108.0,7308184.0,659613.0,1780955.0,1329870.0,429748.0,7600.0,107.0,11734.0,1095.0,557234.0,98433.0,841.0,34.0,2752.0,180.0,17793.0,NaN,NaN,1426574.0,3886148.0,183867.0,69881.0,1732138.0,61151.0,450284.0,102239.0,6.0,15.0,84.0,18.0,736.0,22.71,93.0,11.0,4.0,0.0,1.0,2.0
2,AGV,Aetsveldse Polder Oost,-1.44,8547763.0,8345713.0,762841.0,3130037.0,505074.0,2598187.0,907.0,15907.0,5184.0,4696.0,240775.0,517686.0,207.0,1897.0,1538.0,694.0,23553.0,-2.20,0.76,579024.0,6468111.0,5981.0,59852.0,1193033.0,238405.0,2624881.0,522021.0,31.0,68.0,32.0,6.0,855.0,19.89,87.0,40.0,15.0,27.0,0.0,2.0
3,AGV,Aetsveldse Polder west,-1.45,2777844.0,2729345.0,245893.0,1076461.0,178588.0,892901.0,190.0,0.0,38.0,4741.0,92700.0,152619.0,18.0,0.0,9.0,546.0,9668.0,-2.25,0.80,228365.0,2397425.0,9974.0,286.0,1809.0,138986.0,897870.0,153192.0,32.0,62.0,38.0,6.0,278.0,17.06,78.0,37.0,2.0,0.0,2.0,3.0
4,AGV,Aetsveldse Polder west (Driemond),-0.75,67417.0,67417.0,6384.0,21204.0,994.0,7189.0,0.0,695.0,12308.0,0.0,1046.0,3344.0,0.0,126.0,1865.0,0.0,182.0,NaN,NaN,992.0,16656.0,0.0,1766.0,46839.0,1194.0,20192.0,5335.0,30.0,84.0,16.0,1.0,7.0,26.42,100.0,43.0,0.0,39.0,26.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126,AGV,Westerpark,0.31,63252.0,63252.0,5830.0,20010.0,3231.0,14549.0,0.0,0.0,2230.0,0.0,2733.0,2786.0,0.0,0.0,311.0,0.0,11.0,NaN,NaN,3229.0,44100.0,0.0,30.0,15972.0,0.0,16779.0,3097.0,27.0,53.0,47.0,5.0,6.0,18.46,100.0,33.0,0.0,0.0,14.0,0.0
127,AGV,Wiel Onderwal,-0.22,35885.0,35885.0,3316.0,10848.0,9206.0,1642.0,0.0,0.0,0.0,0.0,3163.0,153.0,0.0,0.0,0.0,0.0,15.0,NaN,NaN,9633.0,24045.0,0.0,9.0,2154.0,0.0,1642.0,153.0,5.0,5.0,95.0,26.0,4.0,9.31,96.0,7.0,0.0,0.0,0.0,0.0
128,AGV,Zuid Bijlmer,-2.06,8377540.0,8313652.0,771722.0,1651983.0,1126297.0,501920.0,514.0,5807.0,15314.0,1796.0,659999.0,108619.0,100.0,570.0,2133.0,242.0,988.0,-2.70,0.64,1167264.0,4308130.0,7969.0,36330.0,2664564.0,185799.0,525351.0,111664.0,6.0,14.0,86.0,13.0,838.0,21.26,96.0,12.0,6.0,16.0,1.0,1.0
129,AGV,Zuider Legmeerpolder,-4.88,8990851.0,8945393.0,814672.0,2198567.0,510732.0,567369.0,669890.0,382733.0,64278.0,3401.0,386430.0,210328.0,139786.0,58746.0,18220.0,1078.0,12486.0,-5.97,1.09,702554.0,2161531.0,1008321.0,3163682.0,1884985.0,67443.0,1687672.0,428159.0,19.0,53.0,47.0,6.0,899.0,25.37,73.0,26.0,66.0,12.0,3.0,5.0


### 3.3 Controleer missende polders 

**Doel**  
Optioneel kan je de originele shapefile die je hebt gebruikt om de rekengebieden de genereren hier invoeren om te checken welke polders niet in de dataset voorkomen.

In [1]:
import arcpy
import pandas as pd

def find_missing_polders(
    shapefile,
    df_all,
    shp_fld_waterschap="Waterschap",
    shp_fld_polder="Naam_1"
):

    shp_set = {
        (str(ws).strip(), str(pol).strip())
        for ws, pol in arcpy.da.SearchCursor(
            shapefile,
            [shp_fld_waterschap, shp_fld_polder]
        )
    }

    df_set = {
        (str(ws).strip(), str(pol).strip())
        for ws, pol in zip(
            df_all["waterschap"],
            df_all["polder"]
        )
    }

    missing = shp_set - df_set

    return pd.DataFrame(
        sorted(missing),
        columns=["waterschap", "polder"]
    )

In [3]:
df_hhr_all = pd.read_csv(r"D:\04_results\hhr_results\df_hhr_bui92mm.csv")

df_missing = find_missing_polders(
    shapefile=r"C:\Users\Senden02\OneDrive - Waternet Amsterdam\Documenten\ArcGIS\Projects\RvW-DPCH\rvw_tool\01_src\03_data_processing\ahn_merger\ahn_merger.gdb\afvoergebieden_compleet_correct_dp_hout_hhr",
    df_all=df_hhr_all
)

df_missing

,waterschap,polder
0,Rijnland,De Verdolven Landen
1,Rijnland,Haarlemmermeerpolder
2,Rijnland,Hogergelegen Santpoort
3,Rijnland,Inmaling Duinland
4,Rijnland,Inmaling Duinrell
5,Rijnland,Kadebuurt
6,Rijnland,Kikkerpolder
7,Rijnland,Landgoed De Paauw
8,Rijnland,Landgoed de Wittenburg
9,Rijnland,Lentevreugd


In [18]:
df_agv_kloten = pd.read_csv(r"D:\04_results\27072026\agv_results\df_agv_bui92mm.csv", sep=";")
df_agv_kloten["sum_kuub"] = df_agv_kloten["water_m3"] + df_agv_kloten["grasland_m3"] + df_agv_kloten["akker_m3"] + df_agv_kloten["tuinbouw_m3"] + df_agv_kloten["bebouwing_binnen_m3"] + df_agv_kloten["bebouwing_buiten_m3"] 
df_agv_kloten["kuub_verschil"] = df_agv_kloten["volume_m3"] - df_agv_kloten["sum_kuub"]
pd.set_option("display.max_rows", None)
df_agv_kloten

,waterschap,polder,WL,opp_pldr_m2,opp_pg_m2,volume_m3,inundatie_m2,water_m2,grasland_m2,akker_m2,tuinbouw_m2,bebouwing_binnen_m2,bebouwing_buiten_m2,water_m3,grasland_m3,akker_m3,tuinbouw_m3,bebouwing_binnen_m3,bebouwing_buiten_m3,abs_diff,rvw_m2,rvw_m3,rvw_m2_Proc,rvw_m3_Proc,sloot_Proc,water_m2_Proc,opp_pldr_ha,gem_inun_m,sum_kuub,kuub_verschil
0,AGV,'s-Gravelandsche Polder,4.800000e-01,8.480336e+06,6.034040e+06,785685.2,2630535.2,508357.5,1933006.0,20512.0,66539.8,92863.0,8151.8,307300.7,445745.0,3072.7,15918.4,13999.3,1030.3,5.494287e+03,2121072.6,479765.7,25.011658,61.063350,39.112446,5.994544,848.033601,22.619014,787066.4,-1.381200e+03
1,AGV,'s-Gravelandsche vaartboezem,1.600000e-01,7.363108e+06,7.308184e+06,660065.3,1780954.8,1329870.4,429747.7,7600.2,107.0,11734.0,1095.0,557829.9,98419.1,841.0,33.9,2751.6,179.9,1.734063e+04,450283.9,102225.5,6.115405,15.487180,84.511320,18.061265,736.310793,22.702455,660055.4,9.900000e+00
2,AGV,Aetsveldse Polder Oost,-1.280000e+00,8.547763e+06,8.458780e+06,780776.3,4247785.0,528835.7,3667937.8,1097.2,23203.7,11898.8,14718.2,321776.4,722057.4,367.0,5099.5,2837.1,2130.2,5.617895e+03,3718855.7,732491.2,43.506771,93.815757,41.212368,6.186832,854.776299,19.696683,1054267.6,-2.734913e+05
3,AGV,Aetsveldse Polder west,-1.450000e+00,2.777844e+06,2.729345e+06,246614.4,1076461.2,178587.6,892900.7,190.0,0.0,37.8,4741.0,92650.3,143672.8,18.0,0.0,8.7,546.5,8.947292e+03,897869.5,144246.0,32.322526,58.490502,37.568893,6.428999,277.784448,16.065364,236896.3,9.718100e+03
4,AGV,Aetsveldse Polder west (Driemond),-7.500000e-01,6.741670e+04,6.741670e+04,6384.1,21204.2,993.5,7188.8,0.0,695.2,12308.2,0.0,1045.5,3344.6,0.0,125.8,1864.4,0.0,1.817635e+02,20192.2,5334.8,29.951332,83.563854,16.376623,1.473670,6.741670,26.420103,6380.3,3.800000e+00
5,AGV,Atekpolder,-7.800000e-01,2.587904e+04,2.587904e+04,2384.5,6242.0,725.8,2954.8,0.0,0.0,2561.5,0.0,660.7,997.6,0.0,0.0,725.7,0.0,3.628737e+00,5516.3,1723.3,21.315709,72.270916,27.708115,2.804587,2.587904,31.240143,2384.0,5.000000e-01
6,AGV,B.O.B.M.-polder en Buitendijken tussen Muiderberg,-5.200000e-01,2.684487e+06,2.644617e+06,243204.1,970047.8,222515.7,734526.7,8448.2,27.4,2669.0,1849.9,108276.2,145401.9,616.4,7.4,755.9,277.3,3.768727e+03,747521.2,147058.9,27.845958,60.467278,44.520713,8.288946,268.448725,19.672873,255335.1,-1.213100e+04
7,AGV,BP Huis Te Vraag,-1.000000e-01,4.193783e+04,4.193783e+04,3850.5,14622.2,3630.8,9118.5,0.0,21.0,1849.0,0.0,1474.9,2196.1,0.0,1.1,177.5,0.0,7.780201e+00,10988.5,2374.7,26.201881,61.672510,38.304116,8.657578,4.193783,21.610775,3849.6,9.000000e-01
8,AGV,Baambrugge Oostzijds,-1.850000e+00,7.252899e+06,7.370530e+06,622592.5,2709585.3,640580.1,2059852.7,3370.4,10.7,580.2,5178.3,281142.3,332099.7,239.7,2.9,79.9,548.3,4.467422e+04,2068992.3,332970.5,28.526418,53.481290,45.156712,8.832056,725.289912,16.093366,614112.8,8.479700e+03
9,AGV,Baambrugge Oostzijds (west),-1.910000e+00,1.393604e+06,8.189574e+05,125692.2,536055.8,81011.5,437493.8,12.5,96.5,0.0,17379.5,46964.8,81669.6,0.8,12.0,0.0,1953.4,2.519412e+03,454982.3,83635.8,32.647879,66.540167,37.364928,5.813091,139.360448,18.382210,130600.6,-4.908400e+03


### 4.1 Histogrammen

In [65]:
import matplotlib.pyplot as plt

def plot_histogram(
    df,
    col,
    title=None,
    xlabel=None,
    bins=100,
    figsize=(6, 5),
    xmin=None,
    xmax=None
):

    fig, ax = plt.subplots(figsize=figsize)

    df[col].hist(
        bins=bins,
        edgecolor="black",
        ax=ax
    )

    ax.set_title(title or col)
    ax.set_xlabel(xlabel or col)
    ax.set_ylabel("Aantal polders")

    if xmin is not None or xmax is not None:
        ax.set_xlim(xmin, xmax)

    plt.tight_layout()
    plt.show()

In [67]:
plot_histogram(
    df_hhr_rvw,
    "abs_diff",
    title="abs_diff (m3) HHR",
    xlabel="abs_diff (m3)",
    xmin=0,
    xmax=650000
)

In [72]:
import matplotlib.pyplot as plt

def plot_histograms_waterschappen(
    dfs,
    namen,
    col,
    bins=20,
    figsize=(14, 10),
    xmin=None,
    xmax=None,
    sharex=True,
    sharey=True
):
    """
    Plot dezelfde variabele voor meerdere waterschappen.

    Parameters
    ----------
    dfs : list
        Lijst met dataframes (of None).
    namen : list
        Namen van de waterschappen.
    col : str
        Te plotten kolom.
    """

    fig, axes = plt.subplots(
        2,
        2,
        figsize=figsize,
        sharex=sharex,
        sharey=sharey
    )

    axes = axes.flatten()

    for ax, df, naam in zip(axes, dfs, namen):

        if df is None:
            ax.set_title(naam)
            ax.text(
                0.5,
                0.5,
                "Nog niet beschikbaar",
                ha="center",
                va="center",
                fontsize=12
            )
            continue

        df[col].hist(
            bins=bins,
            edgecolor="black",
            ax=ax
        )

        ax.set_title(naam)
        ax.set_xlabel(col)
        ax.set_ylabel("Aantal polders")

        if xmin is not None or xmax is not None:
            ax.set_xlim(xmin, xmax)

    plt.tight_layout()
    plt.show()

In [77]:
plot_histograms_waterschappen(
    dfs=[
        None,          # HHNK rekent nog
        df_hhr_rvw,
        df_hdsr_rvw,
        df_agv_rvw
    ],
    namen=[
        "HHNK",
        "HHR",
        "HDSR",
        "AGV"
    ],
    col="rvw_m2_%",
    bins=20,
    xmin=0,
    xmax=90
)

In [161]:
import matplotlib.pyplot as plt

def plot_waterschap_dashboard(
    df,
    waterschap_naam,
    bins=30
):
    """
    3x3 dashboard met histogrammen voor één waterschap.
    """

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(15, 12)
    )
    
    axes = axes.flatten()

    variabelen = [
        ("sloot_%", "% buivolume in sloot (m3) t.o.v. totale bui (m3)"),
        ("rvw_m3_%", "% RvW op maaivled (m3) t.o.v. totale bui (m3)"),
        ("rvw_m2_%", "% RvW op maaiveld (m2) t.o.v. polderoppervlak (m2)"),
#         ("peilstijging_cm", "Peilstijging (cm)"),
        ("water_m2_%", "% water (m2) t.o.v. polderoppervlak (m2)"),
        ("opp_pldr_ha", "Polderoppervlak (ha)"),
#         ("bebouwing_pct", "% bebouwing"),
        ("gem_inun_m", "Gem. inundatiediepte (cm)"),
#         ("p90_diepte_cm", "P90 inundatiediepte (cm)")
    ]

    for ax, (kolom, titel) in zip(axes, variabelen):

        if kolom not in df.columns:
            ax.text(
                0.5,
                0.5,
                f"Kolom ontbreekt:\n{kolom}",
                ha="center",
                va="center"
            )
            ax.set_title(titel)
            continue

        df[kolom].hist(
            bins=bins,
            edgecolor="black",
            ax=ax
        )

        ax.set_title(titel)
        ax.set_ylabel("Aantal polders")

    fig.suptitle(
        f"Waterschap {waterschap_naam}",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()

In [168]:
plot_waterschap_dashboard(
    df_hdsr_rvw,
    "HDSR"
)
